In [1]:
import os
import json
import pandas as pd

In [2]:
base_dir = "../Data_Sailnjord/Hyères November 2025/Straight_lines"
summary_file = "summary.json"
output_file = "summary_enriched.json"


In [3]:
def load_interview_data(interview_dir):
    name_map = {
        "Gian": "Gian Stragiotti",
        "Karl": "Karl Maeder",
        "SenseBoard": "SenseBoard",
        "Max": "Max Maeder"
    }

    dfs = []
    for file in os.listdir(interview_dir):
        key = file.replace("Interview ", "").replace(".xlsx", "").split()[0]
        name = name_map.get(key, key)
        df = pd.read_excel(os.path.join(interview_dir, file))
        df["Name"] = name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def get_boat_info(df, boat_name, run_idx, leg_idx, date_folder=None, interval_start_time=None):
    candidates = df[
        (df["Name"] == boat_name) &
        (df["Run"].str.replace("Run", "", regex=False).astype(int) == run_idx + 1) &
        (df["Leg U=1, D=2"] == leg_idx + 1)
    ].copy()
    row = candidates.iloc[0]
    brand_map = {0: "Levi", 1: "Chub"}
    raw_brand = row.get("Mast brand (0=Levi,1=Chub)", None)
    mast_brand = brand_map.get(int(raw_brand)) if pd.notnull(raw_brand) else None
    load_cell_availability = int(row.get("Load cells (No = 0, Yes = 1, Half=2)"))

    return {
        "total_weight": row.get("Total weight", None),
        "master_leeward": bool(row.get("Master leeward (1)", False)),
        "mast_brand": mast_brand,
        "load_cell_availability": load_cell_availability
    }


In [4]:
"""with open(summary_file, "r") as f:
    summary_data = json.load(f)

for date_folder in sorted(os.listdir(base_dir)):
    date_path = os.path.join(base_dir, date_folder)
    interview_dir = os.path.join(date_path, "Interview and equipment")
    print(interview_dir)
    interview_df = load_interview_data(interview_dir)
    for run in summary_data:
        if run["run"].startswith(date_folder):
            run_number = int(run["run"].split("_Run")[1]) - 1
            for leg_idx, interval in enumerate(run["intervals"]):
                for b in [1, 2]:
                    boat = interval.get(f"boat{b}_name", "")
                    info = get_boat_info(interview_df, boat, run_number, leg_idx, date_folder)
                    interval[f"boat{b}_total_weight"] = info["total_weight"]
                    interval[f"boat{b}_master_leeward"] = info["master_leeward"]
                    interval[f"boat{b}_mast_brand"] = info["mast_brand"]
                    interval[f"boat{b}_load_cell_availability"] = info["load_cell_availability"]


with open(output_file, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"✅ Résumé enrichi sauvé dans {output_file}")"""

with open(summary_file, "r") as f:
    summary_data = json.load(f)
# Scan de tous les dossiers de date
target_folders = ["26_11_2025", "27_11_2025"]

# On filtre pour ne garder que les dossiers présents qui sont dans notre liste
for date_folder in sorted([f for f in os.listdir(base_dir) if f in target_folders]):
    date_path = os.path.join(base_dir, date_folder)
    interview_dir = os.path.join(date_path, "Interview and equipment")
    print(interview_dir)

    interview_df = load_interview_data(interview_dir)

    for run in summary_data:
        if not run["run"].startswith(date_folder):
            continue

        run_number = int(run["run"].split("_Run")[1]) - 1

        for leg_idx, interval in enumerate(run["intervals"]):

            # ======================================================
            # ✅ UPWIND / DOWNWIND CLASSIFICATION (boat1 ONLY)
            # ======================================================
            twa1 = interval.get("avg TWA boat1", None)

            if twa1 is not None:
                interval["mean_TWA"] = twa1
                interval["leg_type"] = "upwind" if abs(twa1) < 90 else "downwind"
            else:
                interval["mean_TWA"] = None
                interval["leg_type"] = "unknown"

            # ======================================================
            # Existing interview / equipment enrichment
            # ======================================================
            for b in [1]:
                boat = interval.get(f"boat{b}_name", "")
                info = get_boat_info(
                    interview_df,
                    boat,
                    run_number,
                    leg_idx,
                    date_folder
                )

                interval[f"boat{b}_total_weight"] = info["total_weight"]
                interval[f"boat{b}_master_leeward"] = info["master_leeward"]
                interval[f"boat{b}_mast_brand"] = info["mast_brand"]
                interval[f"boat{b}_load_cell_availability"] = info["load_cell_availability"]

with open(output_file, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"✅ Résumé enrichi sauvé dans {output_file}")


../Data_Sailnjord/Hyères November 2025/Straight_lines\26_11_2025\Interview and equipment


../Data_Sailnjord/Hyères November 2025/Straight_lines\27_11_2025\Interview and equipment
✅ Résumé enrichi sauvé dans summary_enriched.json


In [5]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator

sog_ref_path = "../Data_Sailnjord/lowess_postprocessed.csv" 
sog_ref_df = pd.read_csv(sog_ref_path, index_col=0)
sog_ref_df.head()

,0.00,0.72,1.45,2.17,2.89,3.61,4.34,5.06,5.78,6.51,...,173.49,174.22,174.94,175.66,176.39,177.11,177.83,178.55,179.28,180.00
Intensità_Vento,,,,,,,,,,,,,,,,,,,,,
6.00,21.005118,21.003750,21.001975,20.999811,20.997306,20.994520,20.991506,20.988293,20.984885,20.981255,...,25.526389,25.536981,25.546632,25.555435,25.563428,25.570593,25.576869,25.582181,25.586477,25.589764
6.06,21.008815,21.007325,21.005401,21.003071,21.000392,20.997436,20.994265,20.990918,20.987405,20.983705,...,25.533363,25.543887,25.553518,25.562336,25.570368,25.577587,25.583929,25.589312,25.593679,25.597032
6.11,21.013892,21.012241,21.010123,21.007573,21.004663,21.001476,20.998089,20.994553,20.990884,20.987074,...,25.542421,25.552812,25.562361,25.571132,25.579141,25.586355,25.592701,25.598095,25.602479,25.605849
6.17,21.020403,21.018566,21.016220,21.013408,21.010217,21.006745,21.003084,20.999298,20.995420,20.991448,...,25.553538,25.563713,25.573099,25.581743,25.589651,25.596782,25.603059,25.608398,25.612737,25.616074
6.22,21.028260,21.026229,21.023643,21.020553,21.017054,21.013261,21.009280,21.005193,21.001048,20.996860,...,25.566554,25.576420,25.585550,25.593979,25.601702,25.608673,25.614811,25.620030,25.624273,25.627536


In [6]:

sog_ref_df.index = pd.to_numeric(sog_ref_df.index, errors="raise")
sog_ref_df.columns = pd.to_numeric(sog_ref_df.columns, errors="raise")

# rebuild arrays + interpolator AFTER this
tws_vals = sog_ref_df.index.values
twa_vals = sog_ref_df.columns.values
Z = sog_ref_df.values

interp = RegularGridInterpolator(
    (tws_vals, twa_vals),
    Z,
    bounds_error=True,
    method="linear"
)

In [7]:
# %%
import numpy as np

TWS = 6.0

# convert column labels once (no forcing, just numeric view)
twa_cols = sog_ref_df.columns.astype(float).values

# find two nearest TWA grid values around your range
TWA1 = twa_cols[np.argmin(np.abs(twa_cols - 1.45))]
TWA2 = twa_cols[np.argmin(np.abs(twa_cols - 2.17))]
TWA_mid = 0.5 * (TWA1 + TWA2)

# table values
sog_1 = sog_ref_df.loc[TWS, TWA1]
sog_2 = sog_ref_df.loc[TWS, TWA2]

# manual midpoint
manual_mid = 0.5 * (sog_1 + sog_2)

# scipy interpolation
scipy_mid = interp([[TWS, TWA_mid]])[0]

print("TWA grid used :", TWA1, TWA2)
print("Manual midpoint:", manual_mid)
print("SciPy midpoint :", scipy_mid)
print("Difference     :", scipy_mid - manual_mid)


TWA grid used : 1.45 2.17
Manual midpoint: 21.000893341285785
SciPy midpoint : 21.000893341285785
Difference     : 0.0


In [8]:


for run in summary_data:
    for interval in run["intervals"]:
        for b in [1]:
            avg_twa = abs(interval[f"avg TWA boat{b}"])
            avg_tws = interval[f"avg TWS boat{b}"]

            sog_for_ref = min(max(avg_tws, 6), 20)

            sog_ref = interp([[sog_for_ref, avg_twa]])[0]

            interval[f"boat{b}_SOG_ref"] = float(sog_ref)
            avg_sog = interval[f"avg_SOG_boat{b}"]
            interval[f"boat{b}_pol_ratio"] = avg_sog / sog_ref * 100
with open(output_file, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"summary_enriched mis à jour : {output_file}")

summary_enriched mis à jour : summary_enriched.json
